In [1]:
# Skeleton Training Prototype
# ===========================
#
# - This is the jupyter notebook I created to define and test out the training process for Skeleton -> moved to google colab
#     - a cross-attention transformer that predicts the next coordinate (3D vector) relative to a ligand centroid given its molecular fingerprint
# - You can run this notebooks's cells yourself, but just be aware the model that gets produced is watered down and its almost definitely going to output coordinates that oscillate around the same area.
#     - Compared to the google colab file, it has half the embedding size, much lower batch size, less heads and extremely restricted training data (compared to the total 14M+)
#     - still fun to mess around with though, I was able to get some prototypes going that could kind of grasp the general pattern
#     - also included some small tests on it to see if it was learning anything (checking output for different molecules, coordinate output statistics)

In [2]:
import torch
import pandas as pd
%matplotlib inline
from pathlib import Path

In [3]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

data_folder = Path
RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
TRAIN_POINTERS = pd.read_parquet(DATAPATH / "training_pointers.parquet")
TEST_POINTERS = pd.read_parquet(DATAPATH / "test_pointers.parquet")

In [4]:
print(MOLECULAR_FINGERPRINTS.head(3))
print(RADIAL_SEQUENCES.head(3))

sample_fp = MOLECULAR_FINGERPRINTS["map4_fp"].iloc[0]
print(len(sample_fp), sample_fp[0].dtype)

                                          smiles_str  \
0  C[C@@H](N1CC[C@](C)(C1=O)c1ccc(OCc2cc(C)nc3ccc...   
1  Cc1ccc(CNc2cc(OC[C@H]3C[C@@H]3c3ccc4CCCc4n3)nn...   
2  C[C@@H](COc1ccnc2CCC[C@@H](C)c12)C[C@H]1Cc2cc3...   

                                             map4_fp  
0  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
1  [tensor(0.), tensor(1.), tensor(0.), tensor(0....  
2  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
  PDB_ID                                    radial_sequence
0   5UEY  [[[0.13, 0, 5.98, 100, 20, 0, 0, 0, 0, 0, 0, 0...
1   6FID  [[[0.18, 0, 5.66, 63, 2.8, 0, 0, 0, 0, 0, 0, 0...
2   1WCC  [[[0.13, 0, 5.98, 97, 5.5, 0, 0, 0, 0, 0, 0, 0...
1024 torch.float32


In [5]:
training_pointer = TRAIN_POINTERS.sample(10000)
test_pointer = TEST_POINTERS.sample(3000)

In [18]:
from torch.utils.data import Dataset

class RadialDataset(Dataset):

    def __init__(self, pointer, smiles_molfp, pdb_radial,
                 block_size, max_angstroms):
        self.pointer = pointer
        self.smiles_molfp = smiles_molfp.set_index("smiles_str")       # <class 'torch.Tensor'>
        self.pdb_radial = pdb_radial.set_index("PDB_ID")               # <class 'list'>
        # set index moves "column label" column to row label, enabling O(1) .loc["target_ID"] lookups

        self.block_size = block_size
        self.max_angstroms = max_angstroms

    def __len__(self):
        return len(self.pointer)


    def __getitem__(self, idx):
        row = self.pointer.iloc[idx]
        smiles = row["SMILES"]
        pdb_id = row["PDB_HIT"]
        target_idx = row["WINDOW_IDX"]

        # context query
        molecular_fingerprint = self.smiles_molfp.loc[smiles, "map4_fp"]

        # radial sequence query
        radial_sequence = self.pdb_radial.loc[pdb_id, "radial_sequence"]

        radial_X, radial_Y = self.radial_XY(radial_sequence, target_idx)

        return (molecular_fingerprint,
                torch.tensor(radial_X, dtype=torch.float32),     # block_size, 3
                torch.tensor(radial_Y, dtype=torch.float32))     # 3,


    def radial_XY(self, radial_seq, target_idx):
        """
        Generates X and Y set for a given radial sequence + molecular fingerprint in the background?
        """
        r_pad = int(self.max_angstroms)  # max range + 1
        az_pad = 0
        pl_pad = 0
        context = [[r_pad, az_pad, pl_pad] for _ in range(self.block_size)]
        for idx in range(len(radial_seq)):
            coords = radial_seq[idx][1]  # = [X, Y, Z]
            if idx == target_idx:
                return context, coords  # This is our XY data
            context = context[1:]+[coords]  # sliding window append -> context style in nanogpt

In [19]:
# RadialSeeker parameters used -> record these - might design a config file later
max_angstroms = 33

block_size = 75
batch_size = 64
n_embd = 256
n_head = 4
n_layers = 4
map4_res = 1024
dropout=0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
loss_alpha = 0.5
n_epochs = 5

# training dataset
training_dataset = RadialDataset(pointer=training_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms)

# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

torch.manual_seed(311104)

# test / validation set
test_set = RadialDataset(pointer=test_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms)
# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=4,
    shuffle=False,
    drop_last=True
)

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# SKELETON MODEL
class SkeletonModel(nn.Module):
    def __init__(self, n_embd, n_head, n_layers, block_size,
                 map4_res, max_angstroms,
                dropout):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.max_angstroms = max_angstroms

        self.c_project = nn.Linear(3, n_embd)  # feed coordinates here
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_encoder = MolecularEncoder(n_embd, map4_res, dropout)


        self.stacks = nn.ModuleList([Stack(n_embd, n_head, block_size, dropout) for _ in range(n_layers)])

        # OUTPUT HEAD -> outputs coordinates
        self.ln_f = nn.LayerNorm(n_embd)
        self.c_head = nn.Linear(n_embd, 3)


    def forward(self, coord_hist, map4_enc, targets=None):
        """
        coord_hist will be [n_batch, block_size, 3]
        targets is [n_batch, 1, 3]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = coord_hist.shape
        coordinate_emb = self.c_project(coord_hist.float())
        pos_emb = self.pos_emb(torch.arange(T, device=coord_hist.device))
        x = coordinate_emb + pos_emb

        mol_emb = self.mol_encoder(map4_enc.float()).unsqueeze(1)

        for stack in self.stacks:
            x = stack(x, mol_emb)

        r, az, pl = self.c_head(self.ln_f(x[:, -1, :])).unbind(dim=1)
        r = F.softplus(r)                 # clip off negative values
        az = torch.tanh(az) * torch.pi    # prevent outputs from being nonsensical angles
        pl = torch.sigmoid(pl) * torch.pi

        output = torch.stack([r, az, pl], dim=1)

        # circular loss
        loss = None
        if targets is not None:
            Xrad, Xazm, Xplr = output.unbind(dim=1)
            Yrad, Yazm, Yplr = targets.unbind(dim=1)

            radial_loss = F.mse_loss(Xrad, Yrad)  # linear scale
            azm_loss = self.circle_loss(Xazm, Yazm)
            pol_loss = self.circle_loss(Xplr, Yplr)

            loss = (radial_loss * 0.5) + (azm_loss * 0.25) + (pol_loss * 0.25)

        return output, loss

    def circle_loss(self, aX, aY):
        return ((torch.cos(aX) - torch.cos(aY))**2 + (torch.sin(aX) - torch.sin(aY))**2).mean()


    def generate(self, map4_enc, max_AAs=130):
        # largest protein pocket in dataset was 107
        map4_enc = map4_enc.to(next(self.parameters()).device)
        coord_context = torch.tensor([[self.max_angstroms * 1.5, 0, 0] for _ in range(self.block_size)]).unsqueeze(0).to(map4_enc.device)

        coord_out = []

        for _ in range(max_AAs):
            next_coord, _ = self.forward(coord_context, map4_enc)
            coord_out.append(next_coord.detach())
            coord_context = torch.cat([coord_context[:, 1:, :], next_coord.unsqueeze(1)], dim=1)

        return torch.stack(coord_out, dim=1).squeeze(0)


class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # added in a causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class FeedForwardNet(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Stack(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForwardNet(n_embd, dropout)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head, dropout)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x


class MolecularEncoder(nn.Module):
    def __init__(self, n_embd, map4_res, dropout):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Linear(map4_res, map4_res),
            nn.LayerNorm(map4_res),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check1 = nn.Linear(map4_res, map4_res // 2)

        self.block2 = nn.Sequential(
            nn.Linear(map4_res // 2, map4_res // 2),
            nn.LayerNorm(map4_res // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check2 = nn.Linear(map4_res // 2, n_embd)
        self.residual1 = nn.Linear(map4_res, map4_res // 2)

        self.block3 = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.LayerNorm(n_embd),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.residual2 = nn.Linear(map4_res // 2, n_embd)


        self.output = nn.LayerNorm(n_embd)

    def forward(self, x):
        x1 = self.block1(x) + x
        x1_down = self.check1(x1)

        x2 = self.block2(x1_down) + self.residual1(x1)
        x2_down = self.check2(x2)

        x3 = self.block3(x2_down) + self.residual2(x2)

        return self.output(x3)

In [21]:
model = SkeletonModel(n_embd=n_embd, n_head=n_head, n_layers=n_layers,
                      block_size=block_size, map4_res=map4_res, max_angstroms=max_angstroms,
                      dropout=dropout).to(device)

In [22]:
learning_rate = 3e-4   # maybe try 5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

SkeletonModel(
  (c_project): Linear(in_features=3, out_features=256, bias=True)
  (pos_emb): Embedding(75, 256)
  (mol_encoder): MolecularEncoder(
    (block1): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
    )
    (check1): Linear(in_features=1024, out_features=512, bias=True)
    (block2): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.1, inplace=False)
    )
    (check2): Linear(in_features=512, out_features=256, bias=True)
    (residual1): Linear(in_features=1024, out_features=512, bias=True)
    (block3): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): G

In [23]:
@torch.no_grad()
def estimate_loss(model, loader, device, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        _, loss = model(radial_X, map4_fp, radial_Y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [24]:
for epoch in range(n_epochs):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        map4_fp = map4_fp.to(device)
        radial_X = radial_X.to(device)
        radial_Y = radial_Y.to(device)

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, max_batches=None)

    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")


Epoch 1 | Batch 1 | Loss: 47.097935
Epoch 1 | Batch 51 | Loss: 2.430671
Epoch 1 | Batch 101 | Loss: 0.983257
Epoch 1 | Batch 151 | Loss: 1.336504
Epoch 1 | Train: 2.756710 | Val: 1.068619 | Gap: -1.688091
Epoch 2 | Batch 1 | Loss: 1.559335
Epoch 2 | Batch 51 | Loss: 0.861494
Epoch 2 | Batch 101 | Loss: 0.985174
Epoch 2 | Batch 151 | Loss: 1.053876
Epoch 2 | Train: 0.991198 | Val: 0.937657 | Gap: -0.053541
Epoch 3 | Batch 1 | Loss: 0.857410
Epoch 3 | Batch 51 | Loss: 0.730391
Epoch 3 | Batch 101 | Loss: 0.911995
Epoch 3 | Batch 151 | Loss: 0.726044
Epoch 3 | Train: 1.005612 | Val: 0.956915 | Gap: -0.048697
Epoch 4 | Batch 1 | Loss: 0.905065
Epoch 4 | Batch 51 | Loss: 0.854258
Epoch 4 | Batch 101 | Loss: 0.803460
Epoch 4 | Batch 151 | Loss: 0.984386
Epoch 4 | Train: 0.921512 | Val: 0.928140 | Gap: 0.006628
Epoch 5 | Batch 1 | Loss: 1.026312
Epoch 5 | Batch 51 | Loss: 1.037518
Epoch 5 | Batch 101 | Loss: 0.938453
Epoch 5 | Batch 151 | Loss: 0.867394
Epoch 5 | Train: 0.927077 | Val: 0.8642

In [25]:
from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"
sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
model.generate(sample_map4)

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_1082430/838496962.py:3: SyntaxWarning: invalid escape sequence '\c'
  sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"


tensor([[ 1.4127e+01, -1.0427e+00,  1.5555e+00],
        [ 1.3976e+01, -1.9501e+00,  1.3995e+00],
        [ 1.3800e+01, -2.4363e+00,  1.5342e+00],
        [ 1.3147e+01, -2.5632e+00,  1.4570e+00],
        [ 1.2878e+01, -2.4914e+00,  1.6305e+00],
        [ 1.3007e+01, -2.4609e+00,  1.5504e+00],
        [ 1.2600e+01, -2.4333e+00,  1.5118e+00],
        [ 1.2415e+01, -2.4394e+00,  1.5231e+00],
        [ 1.2202e+01, -2.5772e+00,  1.4820e+00],
        [ 1.2047e+01, -2.5717e+00,  1.5816e+00],
        [ 1.2059e+01, -2.3959e+00,  1.5051e+00],
        [ 1.2072e+01, -2.2403e+00,  1.5520e+00],
        [ 1.2026e+01, -2.0426e+00,  1.4984e+00],
        [ 1.1909e+01, -2.1526e+00,  1.4157e+00],
        [ 1.1720e+01, -2.1788e+00,  1.4878e+00],
        [ 1.1710e+01, -2.2750e+00,  1.4903e+00],
        [ 1.1586e+01, -2.2019e+00,  1.5115e+00],
        [ 1.1493e+01, -2.2216e+00,  1.5797e+00],
        [ 1.1629e+01, -2.2195e+00,  1.4242e+00],
        [ 1.1741e+01, -2.4495e+00,  1.5331e+00],
        [ 1.1568e+01

In [26]:
# grab a batch and look at target distribution
map4_fp, radial_X, radial_Y = next(iter(loader))
print(f"Target mean: {radial_Y.float().mean():.4f}")
print(f"Target std:  {radial_Y.float().std():.4f}")
print(f"Target min:  {radial_Y.float().min():.4f}")
print(f"Target max:  {radial_Y.float().max():.4f}")

Target mean: 3.7757
Target std:  5.1404
Target min:  -3.0577
Target max:  16.0524


In [27]:
seq_lengths = []
for seq in RADIAL_SEQUENCES["radial_sequence"]:
    seq_lengths.append(len(seq))

seq_lengths = pd.Series(seq_lengths)
print(seq_lengths.describe())
print(f"Median: {seq_lengths.median()}")
print(f"90th percentile: {seq_lengths.quantile(0.90)}")
print(f"95th percentile: {seq_lengths.quantile(0.95)}")

count    8064.000000
mean       46.393973
std        13.658733
min         5.000000
25%        38.000000
50%        48.000000
75%        55.000000
max       108.000000
dtype: float64
Median: 48.0
90th percentile: 63.0
95th percentile: 67.0


In [28]:
model.eval()
with torch.no_grad():
    ctx = torch.zeros(1, block_size, 3, device=device)

    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(ctx, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [0.00023548502940684557, -2.362806558609009, 0.053354497998952866]
Caffeine: [0.00027232043794356287, -2.27482271194458, 0.05536774545907974]
Ibuprofen: [0.0003670541336759925, -2.0922327041625977, 0.059691138565540314]
